# 36 — External validity: Adjusted Rand Index

| | |
|---|---|
| **Author** | Blessy Paul |
| **Contributes to** | `xxcluster/measures/validation/external.py` |
| **Documents** | `documentation/sections/clustering_methods/validity/adjusted_rand.tex` |
| **Dataset** | Iris |

This notebook verifies the implementation of the Adjusted Rand Index (ARI) external clustering validity measure using the Iris benchmark dataset. The implementation is tested by comparing clustering labels with the known Iris class labels.

## 1. Setup

Import the required packages and make the `cluster_analysis` project available to the notebook.

In [1]:
from pathlib import Path
import sys

sys.path.insert(0, str(Path.cwd().parent))

import numpy as np

from sklearn.cluster import KMeans
from sklearn.datasets import load_iris
from sklearn.metrics import adjusted_rand_score

from xxcluster.measures.validation.external import AdjustedRand


## 2. Data

Load the Iris benchmark dataset from scikit-learn. The dataset contains 150 observations, four numerical features, and three known class labels that can be used as the reference partition for external validation.

In [2]:
iris = load_iris()

X = iris.data
y_true = iris.target

print(f"Iris observations: {X.shape[0]}")
print(f"Features: {X.shape[1]}")
print(f"Reference classes: {np.unique(y_true)}")

Iris observations: 150
Features: 4
Reference classes: [0 1 2]


## 3. KMeans clustering

Apply KMeans clustering to the Iris feature data using three clusters to match the number of known Iris classes. The resulting cluster labels will be compared with the reference labels using the Adjusted Rand Index.

In [3]:
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
y_pred = kmeans.fit_predict(X)

print(f"Predicted clusters: {np.unique(y_pred)}")
print(f"Number of predicted labels: {len(y_pred)}")

Predicted clusters: [0 1 2]
Number of predicted labels: 150


## 4. Adjusted Rand Index verification

Calculate the Adjusted Rand Index using the AquaBlend implementation and compare it with scikit-learn's reference implementation. The two scores should match if the implementation behaves correctly.

In [4]:
ari = AdjustedRand()

aquablend_score = ari.score(
    X=X,
    labels=y_pred,
    labels_true=y_true,
    metric=None
)

sklearn_score = adjusted_rand_score(y_true, y_pred)

print(f"AquaBlend Adjusted Rand Index: {aquablend_score:.6f}")
print(f"scikit-learn Adjusted Rand Index: {sklearn_score:.6f}")
print(f"Scores match: {np.isclose(aquablend_score, sklearn_score)}")

AquaBlend Adjusted Rand Index: 0.730238
scikit-learn Adjusted Rand Index: 0.730238
Scores match: True


## 5. Label invariance

Verify that the Adjusted Rand Index is invariant to cluster label permutations. Renaming the predicted cluster identifiers should not change the ARI score.

In [5]:
label_map = {0: 2, 1: 0, 2: 1}
y_pred_relabelled = np.array([label_map[label] for label in y_pred])

original_score = ari.score(
    X=X,
    labels=y_pred,
    labels_true=y_true,
    metric=None
)

relabelled_score = ari.score(
    X=X,
    labels=y_pred_relabelled,
    labels_true=y_true,
    metric=None
)

print(f"Original ARI: {original_score:.6f}")
print(f"Relabelled ARI: {relabelled_score:.6f}")
print(f"Label invariant: {np.isclose(original_score, relabelled_score)}")

Original ARI: 0.730238
Relabelled ARI: 0.730238
Label invariant: True


## 6. Symmetry

Verify that the Adjusted Rand Index is symmetric. Swapping the predicted and reference cluster labels should produce the same ARI score.

In [6]:
score_forward = ari.score(
    X=X,
    labels=y_pred,
    labels_true=y_true,
    metric=None
)

score_reverse = ari.score(
    X=X,
    labels=y_true,
    labels_true=y_pred,
    metric=None
)

print(f"ARI(y_true, y_pred): {score_forward:.6f}")
print(f"ARI(y_pred, y_true): {score_reverse:.6f}")
print(f"Symmetric: {np.isclose(score_forward, score_reverse)}")

ARI(y_true, y_pred): 0.730238
ARI(y_pred, y_true): 0.730238
Symmetric: True


## 7. Perfect agreement

Verify that the Adjusted Rand Index returns 1.0 when the predicted clustering is identical to the reference partition.

In [7]:
perfect_score = ari.score(
    X=X,
    labels=y_true,
    labels_true=y_true,
    metric=None
)

print(f"ARI for identical partitions: {perfect_score:.6f}")
print(f"Perfect agreement: {np.isclose(perfect_score, 1.0)}")

ARI for identical partitions: 1.000000
Perfect agreement: True


## 8. Summary

The AquaBlend Adjusted Rand Index implementation was verified using the Iris benchmark dataset.

- The AquaBlend ARI score matched scikit-learn's reference implementation.
- Relabelling cluster identifiers did not change the ARI score.
- The ARI produced the same result when the two partitions were swapped.
- Identical partitions produced an ARI of 1.0.

These results confirm the expected behaviour of the Adjusted Rand Index implementation on the Iris benchmark dataset.